# LLaMA 3.1 8B + QLoRA — Распознавание эмоций на GoEmotions

**Окружение:** Kaggle Notebooks, GPU T4 (16 GB).  
**Метод:** QLoRA (4-bit NF4 квантование + LoRA-адаптеры).  
**Задача:** мульти-лейбл классификация эмоций (28 классов) на корпусе GoEmotions.

> Перед запуском убедитесь, что включён GPU и Internet в `Settings`, а в `Add-ons → Secrets` сохранён `HF_TOKEN` с доступом к выбранной модели.

## Установка библиотек и базовые импорты

In [ ]:
# Блок 1. Установка библиотек и импорты
# Без пиннинга версий — берём последние совместимые.
# bitsandbytes критически важен: старые версии (0.43.x) ломаются из-за того,
# что в новом triton 3.x убрали модуль triton.ops.
!pip install -q -U bitsandbytes transformers peft accelerate datasets evaluate scikit-learn

import os, time, gc, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorWithPadding, set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, hamming_loss

SEED = 42
set_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")


## Логин в Hugging Face

In [ ]:
# Блок 2. Логин в Hugging Face через Kaggle Secrets
# В Kaggle: Add-ons → Secrets → Add new secret с label `HF_TOKEN`
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("✅ Авторизация в Hugging Face выполнена")


## Параметры эксперимента

In [ ]:
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"
RUN_NAME   = "llama31_8b_qlora_goemotions"


## Данные: GoEmotions

In [ ]:
# Блок 3. Загрузка GoEmotions и multi-hot encoding
dataset = load_dataset("go_emotions", "simplified")
print(dataset)

label_names = dataset['train'].features['labels'].feature.names
num_labels = len(label_names)         # 28 = 27 эмоций + neutral
print(f"Классов: {num_labels}")
print(f"Эмоции: {label_names}")

def multi_hot(batch):
    arr = np.zeros((len(batch['text']), num_labels), dtype=np.float32)
    for i, lbls in enumerate(batch['labels']):
        arr[i, lbls] = 1.0
    return {"encoded_labels": arr.tolist()}

encoded = dataset.map(multi_hot, batched=True, remove_columns=['labels', 'id'])
print("Пример меток первого примера (первые 10 классов):",
      encoded['train'][0]['encoded_labels'][:10])


## Токенизация

In [ ]:
# Блок 4. Токенизация
print(f"Загрузка токенизатора {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 128

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = encoded.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized = tokenized.rename_column("encoded_labels", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(tokenized)


## Загрузка модели в 4-bit (квантование NF4)

In [ ]:
# Блок 5. Загрузка квантованной модели (4-bit NF4) — сердце метода QLoRA

# Tesla T4 (Kaggle free) НЕ поддерживает bf16 → используем fp16
# A100/L4/H100 поддерживают bf16 (более стабильно)
GPU_NAME = torch.cuda.get_device_name(0).lower() if torch.cuda.is_available() else ""
SUPPORTS_BF16 = any(x in GPU_NAME for x in ["a100", "l4", "h100", "rtx 30", "rtx 40", "rtx a"])
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
print(f"GPU: {GPU_NAME} → используем compute_dtype = {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat 4-bit
    bnb_4bit_use_double_quant=True,      # двойное квантование
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

print(f"Загрузка модели {MODEL_NAME} в 4-bit NF4 ...")
t0 = time.time()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
print(f"Модель загружена за {time.time()-t0:.1f} сек")

# Синхронизация pad_token
model.config.pad_token_id = tokenizer.pad_token_id

# Подготовка квантованной модели к обучению (включает gradient checkpointing,
# upcast LayerNorm в fp32 и т.п.)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

torch.cuda.empty_cache()
mem_after_load = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM после загрузки модели: {mem_after_load:.2f} GB")


## Подключение LoRA-адаптеров

In [ ]:
# Блок 6. LoRA-адаптеры поверх квантованной модели = QLoRA
lora_config = LoraConfig(
    r=16,                           # ранг низкоранговых матриц
    lora_alpha=32,                  # scaling factor (по канону alpha = 2*r)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    # Расширенный набор — рекомендация QLoRA paper:
    # лучше адаптировать ВСЕ линейные слои, а не только q_proj/v_proj
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],      # классификационную голову обучаем полностью
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Метрики качества

In [ ]:
# Блок 7. Метрики качества для multi-label классификации
THRESHOLD = 0.5    # порог сигмоиды для бинаризации предсказаний

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))                  # sigmoid
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)

    metrics = {
        "f1_macro":        f1_score(labels, preds, average="macro",    zero_division=0),
        "f1_micro":        f1_score(labels, preds, average="micro",    zero_division=0),
        "f1_weighted":     f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro":    recall_score(labels, preds, average="macro", zero_division=0),
        "hamming_loss":    hamming_loss(labels, preds),
    }
    try:
        metrics["roc_auc_macro"] = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        metrics["roc_auc_macro"] = float("nan")
    return metrics


## Обучение

In [ ]:
# Блок 8. Конфигурация обучения и запуск
OUTPUT_DIR = f"/kaggle/working/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,          # безопасно для T4 16GB
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,          # эффективный batch = 32
    # gradient_checkpointing уже включён через prepare_model_for_kbit_training
    learning_rate=2e-4,                     # типично для LoRA/QLoRA
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="paged_adamw_8bit",               # paged-оптимизатор из QLoRA paper
    bf16=SUPPORTS_BF16,                     # на T4 → False, на A100/L4 → True
    fp16=not SUPPORTS_BF16,                 # на T4 → True
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Замер времени и пиковой памяти
torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
peak_mem = torch.cuda.max_memory_allocated() / 1024**3

print(f"\n⏱  Время обучения: {train_time/60:.2f} мин")
print(f"📈 Пиковая VRAM:    {peak_mem:.2f} GB")


## Оценка на тестовой выборке

In [ ]:
# Блок 9. Финальная оценка на тестовой выборке
test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
print("\n=== Тестовые метрики ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")


## Сохранение адаптера и сводки

In [ ]:
# Блок 10. Сохранение адаптера и сводки эксперимента
adapter_dir = f"{OUTPUT_DIR}/final_adapter"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

def folder_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024**2

adapter_size = folder_size_mb(adapter_dir)
print(f"💾 Размер адаптера: {adapter_size:.2f} MB")

summary = {
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "method": "QLoRA (4-bit NF4)",
    "lora_r": 16,
    "lora_alpha": 32,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    "epochs": training_args.num_train_epochs,
    "effective_batch": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "train_time_min": round(train_time/60, 2),
    "peak_vram_gb": round(peak_mem, 2),
    "adapter_size_mb": round(adapter_size, 2),
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
}
with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n=== ИТОГ ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))
